# Financial Options Pricing Engine

## Black-Scholes, Binomial Trees, Monte Carlo, and Option Greeks

This notebook presents a quantitative engine for pricing European and American financial options.

The project implements and compares:

- Black-Scholes analytical pricing
- Cox-Ross-Rubinstein binomial trees
- Risk-neutral Monte Carlo simulation
- Option Greeks
- Put-call parity
- Sensitivity analysis
- Numerical convergence

The baseline analysis uses European call and put options with a spot price of \$100, a strike price of \$100, one year to maturity, a 5% risk-free rate, 20% annual volatility, and no dividends.

> **Disclaimer:** This project is intended for educational and analytical purposes only. It does not constitute financial or investment advice.

---

**Author:** Alex Carrillo  
**Field:** Financial Mathematics and Quantitative Finance  
**Technology:** Python

## 1. Option Pricing Parameters

An option is a financial contract whose value depends on an underlying asset.

The pricing models use the following inputs:

- $S_0$: current price of the underlying asset
- $K$: strike price
- $T$: time to maturity in years
- $r$: continuously compounded risk-free interest rate
- $\sigma$: annual volatility
- $q$: continuous dividend yield

A **call option** gives its holder the right to buy the asset at the strike price.

A **put option** gives its holder the right to sell the asset at the strike price.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.binomial_tree import (
    binomial_call_price,
    binomial_put_price,
)
from src.black_scholes import (
    OptionParameters,
    black_scholes_call,
    black_scholes_put,
    calculate_put_call_parity_difference,
)
from src.greeks import (
    calculate_call_greeks,
    calculate_put_greeks,
)
from src.monte_carlo import (
    monte_carlo_call_price,
    monte_carlo_put_price,
)

plt.style.use("ggplot")

print(f"Python executable: {sys.executable}")
print(f"Project root: {project_root}")
print("Imports completed successfully.")

Python executable: c:\Users\Admin\Documents\Utech\Proyectos\monte-carlo-financial-simulation\.venv\Scripts\python.exe
Project root: c:\Users\Admin\Documents\Utech\Proyectos\financial-options-pricing-engine
Imports completed successfully.


## 2. Black-Scholes Model

For European options with continuous dividend yield, the Black-Scholes terms are:

$$
d_1 =
\frac{
\ln(S_0/K)
+
\left(
r-q+\frac{\sigma^2}{2}
\right)T
}{
\sigma\sqrt{T}
}
$$

$$
d_2 = d_1-\sigma\sqrt{T}
$$

The European call price is:

$$
C =
S_0e^{-qT}N(d_1)
-
Ke^{-rT}N(d_2)
$$

The European put price is:

$$
P =
Ke^{-rT}N(-d_2)
-
S_0e^{-qT}N(-d_1)
$$

The model assumes constant volatility, constant interest rates, continuous trading, and lognormally distributed asset prices.

In [2]:
parameters = OptionParameters(
    spot_price=100.0,
    strike_price=100.0,
    time_to_maturity=1.0,
    risk_free_rate=0.05,
    volatility=0.20,
    dividend_yield=0.0,
)

parameters

OptionParameters(spot_price=100.0, strike_price=100.0, time_to_maturity=1.0, risk_free_rate=0.05, volatility=0.2, dividend_yield=0.0)

In [3]:
black_scholes_call_value = black_scholes_call(parameters)
black_scholes_put_value = black_scholes_put(parameters)

parity_difference = calculate_put_call_parity_difference(
    parameters
)

black_scholes_results = pd.DataFrame(
    {
        "Option": ["European call", "European put"],
        "Black-Scholes price": [
            black_scholes_call_value,
            black_scholes_put_value,
        ],
    }
)

black_scholes_results.style.format(
    {
        "Black-Scholes price": "${:,.4f}",
    }
)

,Option,Black-Scholes price
0,European call,$10.4506
1,European put,$5.5735


In [4]:
print(f"Put-call parity difference: {parity_difference:.12f}")

Put-call parity difference: 0.000000000000


## 3. Cox-Ross-Rubinstein Binomial Tree

The binomial model divides the option's lifetime into discrete time steps.

During each step, the asset price can move upward or downward:

$$
u=e^{\sigma\sqrt{\Delta t}}
$$

$$
d=\frac{1}{u}
$$

The risk-neutral probability of an upward movement is:

$$
p=
\frac{
e^{(r-q)\Delta t}-d
}{
u-d
}
$$

The option value is calculated backward through the tree:

$$
V_t =
e^{-r\Delta t}
\left[
pV_{t+\Delta t}^{up}
+
(1-p)V_{t+\Delta t}^{down}
\right]
$$

For American options, every node also compares the continuation value with the intrinsic value. This allows the model to represent early exercise.

In [5]:
binomial_steps = 500

binomial_european_call = binomial_call_price(
    parameters=parameters,
    steps=binomial_steps,
    exercise_style="european",
)

binomial_european_put = binomial_put_price(
    parameters=parameters,
    steps=binomial_steps,
    exercise_style="european",
)

binomial_american_call = binomial_call_price(
    parameters=parameters,
    steps=binomial_steps,
    exercise_style="american",
)

binomial_american_put = binomial_put_price(
    parameters=parameters,
    steps=binomial_steps,
    exercise_style="american",
)

binomial_results = pd.DataFrame(
    {
        "Option": [
            "European call",
            "European put",
            "American call",
            "American put",
        ],
        "Binomial price": [
            binomial_european_call,
            binomial_european_put,
            binomial_american_call,
            binomial_american_put,
        ],
    }
)

binomial_results.style.format(
    {
        "Binomial price": "${:,.4f}",
    }
)

,Option,Binomial price
0,European call,$10.4466
1,European put,$5.5695
2,American call,$10.4466
3,American put,$6.0888


### Interpretation

The American call has approximately the same value as the European call because early exercise is generally not optimal for a non-dividend-paying asset.

The American put is more valuable than the European put because the holder may benefit from exercising before maturity.

## 4. Risk-Neutral Monte Carlo Simulation

Under the risk-neutral probability measure, the terminal asset price is simulated as:

$$
S_T =
S_0
\exp
\left[
\left(
r-q-\frac{\sigma^2}{2}
\right)T
+
\sigma\sqrt{T}Z
\right]
$$

where:

$$
Z \sim N(0,1)
$$

The European call and put prices are estimated by discounting the average simulated payoff:

$$
C =
e^{-rT}
E
\left[
\max(S_T-K,0)
\right]
$$

$$
P =
e^{-rT}
E
\left[
\max(K-S_T,0)
\right]
$$

The implementation uses antithetic variates to improve numerical stability and reports the standard error and a 95% confidence interval.

In [6]:
monte_carlo_simulations = 500_000

monte_carlo_call_result = monte_carlo_call_price(
    parameters=parameters,
    simulations=monte_carlo_simulations,
    seed=42,
    antithetic=True,
)

monte_carlo_put_result = monte_carlo_put_price(
    parameters=parameters,
    simulations=monte_carlo_simulations,
    seed=42,
    antithetic=True,
)

monte_carlo_results = pd.DataFrame(
    {
        "Option": [
            "European call",
            "European put",
        ],
        "Estimated price": [
            monte_carlo_call_result.price,
            monte_carlo_put_result.price,
        ],
        "Standard error": [
            monte_carlo_call_result.standard_error,
            monte_carlo_put_result.standard_error,
        ],
        "CI low": [
            monte_carlo_call_result.confidence_interval_low,
            monte_carlo_put_result.confidence_interval_low,
        ],
        "CI high": [
            monte_carlo_call_result.confidence_interval_high,
            monte_carlo_put_result.confidence_interval_high,
        ],
    }
)

monte_carlo_results.style.format(
    {
        "Estimated price": "${:,.4f}",
        "Standard error": "${:,.6f}",
        "CI low": "${:,.4f}",
        "CI high": "${:,.4f}",
    }
)

,Option,Estimated price,Standard error,CI low,CI high
0,European call,$10.4557,$0.020861,$10.4148,$10.4965
1,European put,$5.5738,$0.012260,$5.5498,$5.5979


### Interpretation

Monte Carlo does not produce a single exact analytical value. It produces a statistical estimate.

The Black-Scholes prices lie inside the Monte Carlo confidence intervals, indicating consistency between the analytical and simulation-based methods.

Increasing the number of simulations generally reduces the standard error, although it also increases computational cost.

## 5. Option Greeks

Option Greeks measure how the option price responds to changes in market variables.

- **Delta:** sensitivity to a \$1 change in the underlying asset.
- **Gamma:** sensitivity of Delta to a \$1 change in the underlying asset.
- **Vega:** sensitivity to a one-percentage-point change in volatility.
- **Theta:** approximate change in option value caused by one day passing.
- **Rho:** sensitivity to a one-percentage-point change in the risk-free rate.

Greeks are essential for hedging, portfolio risk management, and sensitivity analysis.

In [7]:
call_greeks = calculate_call_greeks(parameters)
put_greeks = calculate_put_greeks(parameters)

greeks_results = pd.DataFrame(
    {
        "Greek": [
            "Delta",
            "Gamma",
            "Vega (per 1%)",
            "Theta (per day)",
            "Rho (per 1%)",
        ],
        "European call": [
            call_greeks.delta,
            call_greeks.gamma,
            call_greeks.vega,
            call_greeks.theta,
            call_greeks.rho,
        ],
        "European put": [
            put_greeks.delta,
            put_greeks.gamma,
            put_greeks.vega,
            put_greeks.theta,
            put_greeks.rho,
        ],
    }
)

greeks_results.style.format(
    {
        "European call": "{:,.6f}",
        "European put": "{:,.6f}",
    }
)

,Greek,European call,European put
0,Delta,0.636831,-0.363169
1,Gamma,0.018762,0.018762
2,Vega (per 1%),0.375240,0.375240
3,Theta (per day),-0.017573,-0.004542
4,Rho (per 1%),0.532325,-0.418905


### Interpretation of the Greeks

The call Delta indicates that a \$1 increase in the underlying asset produces an approximate \$0.64 increase in the call price, assuming the other variables remain constant.

The put Delta is negative because the put generally loses value when the underlying asset price increases.

Call and put options share the same Gamma and Vega under Black-Scholes. Their Theta values are negative in the baseline case, reflecting time-value decay.

The call has positive Rho, while the put has negative Rho, because higher interest rates affect the present value of the strike price differently.

## 6. Pricing Method Comparison

Black-Scholes provides an analytical benchmark. The binomial tree approximates the analytical value through backward induction, while Monte Carlo estimates it statistically from simulated terminal payoffs.

In [8]:
comparison_results = pd.DataFrame(
    {
        "Method": [
            "Black-Scholes",
            "Binomial (500 steps)",
            "Monte Carlo (500,000 simulations)",
        ],
        "European call": [
            black_scholes_call_value,
            binomial_european_call,
            monte_carlo_call_result.price,
        ],
        "European put": [
            black_scholes_put_value,
            binomial_european_put,
            monte_carlo_put_result.price,
        ],
    }
)

comparison_results["Call absolute error"] = (
    comparison_results["European call"]
    - black_scholes_call_value
).abs()

comparison_results["Put absolute error"] = (
    comparison_results["European put"]
    - black_scholes_put_value
).abs()

comparison_results.style.format(
    {
        "European call": "${:,.4f}",
        "European put": "${:,.4f}",
        "Call absolute error": "${:,.6f}",
        "Put absolute error": "${:,.6f}",
    }
)

,Method,European call,European put,Call absolute error,Put absolute error
0,Black-Scholes,$10.4506,$5.5735,$0.000000,$0.000000
1,Binomial (500 steps),$10.4466,$5.5695,$0.003998,$0.003998
2,"Monte Carlo (500,000 simulations)",$10.4557,$5.5738,$0.005067,$0.000297


### Comparison Summary

All three methods produce consistent European option values.

- Black-Scholes provides the analytical reference value.
- The binomial model approaches Black-Scholes as the number of steps increases.
- Monte Carlo produces a statistical estimate with measurable sampling uncertainty.
- The binomial model can additionally value American options and represent early exercise.
- Monte Carlo is especially useful when analytical solutions are unavailable or the payoff structure becomes more complex.

## 7. Visual Analysis

### Option Payoffs at Maturity

The payoff diagram shows the asymmetric rights provided by call and put options.

![Option payoffs](../results/figures/option_payoffs.png)

### Binomial Convergence

The binomial prices approach the Black-Scholes analytical values as the number of time steps increases.

![Binomial convergence](../results/figures/binomial_convergence.png)

### Spot Price Sensitivity

Call prices generally increase with the underlying asset price, while put prices generally decrease.

![Spot price sensitivity](../results/figures/spot_price_sensitivity.png)

### Volatility Sensitivity

Both call and put prices generally increase as volatility rises because greater uncertainty increases the potential value of the option payoff.

![Volatility sensitivity](../results/figures/volatility_sensitivity.png)

### Method Comparison

The three methods produce similar European option values under the baseline assumptions.

![Pricing method comparison](../results/figures/method_comparison.png)

## 8. Conclusions

This project developed a quantitative financial options pricing engine using three complementary methods.

Under the baseline assumptions:

- The Black-Scholes European call price is approximately **\$10.4506**.
- The Black-Scholes European put price is approximately **\$5.5735**.
- The 500-step binomial prices are very close to the analytical values.
- The Monte Carlo estimates are consistent with Black-Scholes and include statistical confidence intervals.
- The American call has approximately the same value as the European call when the asset pays no dividends.
- The American put is more valuable than the European put because of the possibility of early exercise.
- Put-call parity is satisfied.
- The Greeks quantify the option's exposure to the underlying asset, volatility, time decay, and interest rates.

### Strengths of Each Method

| Method | Main strength | Main limitation |
|---|---|---|
| Black-Scholes | Fast analytical solution | Restrictive assumptions and European exercise |
| Binomial tree | American exercise and intuitive structure | Computational cost increases with the number of steps |
| Monte Carlo | Flexible for complex stochastic problems | Sampling error and higher computational cost |

### Model Limitations

The models rely on simplifying assumptions:

- Constant volatility
- Constant risk-free interest rate
- Continuous trading
- No transaction costs or taxes
- Lognormally distributed asset prices
- Continuous dividend yield
- Perfectly liquid markets

Real markets may exhibit volatility smiles, jumps, liquidity constraints, changing interest rates, and transaction costs.

### Possible Extensions

Future versions may include:

- Implied volatility
- Volatility surfaces
- Barrier and Asian options
- Stochastic volatility
- Jump-diffusion models
- Finite-difference pricing
- Market-data calibration
- Interactive option-pricing dashboards

> **Disclaimer:** This project is intended exclusively for educational, academic, and analytical purposes. It does not constitute financial or investment advice.